# Notebook: 01_exploracao_corpus_jornalistico.ipynb
**id_rastreabilidade:** RTB_001  
**Versão:** v02  
**Data de criação:** "24/03/2026"  
**Última atualização:** "09/05/2026"  
**Responsável:** Ailton Vendramini  

---

## Finalidade
Explorar e consolidar o corpus jornalístico do Atlas Social de Hortolândia.  
Todos os CSVs estão no schema **v10.4** (20 colunas) após migração completa em mai/2026.

---

## Schema v10.4 — 20 colunas
`id_evento` · `data_publicacao` · `fonte` · `titulo` · `pagina` · `municipio` · `localidade`  
`tipo_camada` · `dimensao_analitica` · `tipo_relacao_variavel` · `codigo_variavel`  
`nivel_criticidade` · `observacao` · `aproximacao_variavel` · `nota_analista`  
`cod_loteamento` · `nivel_confianca_loteamento` · `papel_no_ciclo` · `id_caso_pressao` · `entra_ipst`

---

## Outputs Gerados
| Caminho | tipo_output | Commitar? |
|---|---|---|
| `outputs/tabelas/corpus_consolidado.csv` | exploratorio | Não |

---

## Changelog
- v02 (09/05/2026): migração completa para schema v10.4; remoção de células legacy
- v01 (24/03/2026): exploração inicial (schema antigo)


In [1]:
import pandas as pd
import glob
import os

# ── Configuração ──────────────────────────────────────────────────────────────
PASTA = r'C:\Users\ailto\Atlas-Social-de-Hortolandia\dados\bd_externos\series_jornalisticas'

COLUNAS_V10_4 = [
    'id_evento', 'data_publicacao', 'fonte', 'titulo', 'pagina', 'municipio',
    'localidade', 'tipo_camada', 'dimensao_analitica', 'tipo_relacao_variavel',
    'codigo_variavel', 'nivel_criticidade', 'observacao', 'aproximacao_variavel',
    'nota_analista', 'cod_loteamento', 'nivel_confianca_loteamento',
    'papel_no_ciclo', 'id_caso_pressao', 'entra_ipst'
]

print(f"Pasta: {PASTA}")
print(f"Schema v10.4: {len(COLUNAS_V10_4)} colunas")

Pasta: C:\Users\ailto\Atlas-Social-de-Hortolandia\00_governanca\series_jornalisticas
Schema v10.4: 20 colunas


In [2]:
# ── Carregar todos os CSVs ────────────────────────────────────────────────────
arquivos = sorted(glob.glob(os.path.join(PASTA, "2026_*.csv")))
print(f"Arquivos encontrados: {len(arquivos)}")

dfs = []
ignorados = []

for arq in arquivos:
    try:
        df_tmp = pd.read_csv(arq, sep=",", encoding="utf-8", dtype=str)
        # verifica se tem id_evento (schema v10.4)
        if 'id_evento' not in df_tmp.columns:
            ignorados.append(os.path.basename(arq))
            continue
        # remove coluna peso residual se existir
        df_tmp = df_tmp.drop(columns=['peso'], errors='ignore')
        dfs.append(df_tmp)
    except Exception as e:
        print(f"ERRO em {os.path.basename(arq)}: {e}")

if ignorados:
    print(f"\nIgnorados (schema incompativel): {len(ignorados)}")
    for i in ignorados:
        print(f"  - {i}")

if not dfs:
    raise ValueError("Nenhum arquivo carregado. Verifique o caminho e os schemas.")

df_total = pd.concat(dfs, ignore_index=True, sort=False)

# remove evento politico Clodoaldo (fora do escopo SMIDS)
df_total = df_total[
    ~df_total['titulo'].str.contains("Clodoaldo", na=False)
]

print(f"\n=== BASE CONSOLIDADA ===")
print(f"Total de eventos: {len(df_total)}")
print(f"Colunas: {list(df_total.columns)}")

Arquivos encontrados: 97

=== BASE CONSOLIDADA ===
Total de eventos: 456
Colunas: ['id_evento', 'data_publicacao', 'fonte', 'titulo', 'pagina', 'municipio', 'localidade', 'tipo_camada', 'dimensao_analitica', 'tipo_relacao_variavel', 'codigo_variavel', 'nivel_criticidade', 'observacao', 'aproximacao_variavel', 'nota_analista', 'cod_loteamento', 'nivel_confianca_loteamento', 'papel_no_ciclo', 'id_caso_pressao', 'entra_ipst']


In [3]:
# ── Verificação de integridade ────────────────────────────────────────────────
print("Colunas presentes:")
print(df_total.columns.tolist())

print(f"\nPeriodo coberto:")
print(f"  Início : {df_total['data_publicacao'].min()}")
print(f"  Fim    : {df_total['data_publicacao'].max()}")
print(f"  Edições: {df_total['data_publicacao'].nunique()}")

print(f"\nAmostra (5 registros):")
df_total[['id_evento','data_publicacao','titulo','tipo_camada','nivel_criticidade']].head(5)

Colunas presentes:
['id_evento', 'data_publicacao', 'fonte', 'titulo', 'pagina', 'municipio', 'localidade', 'tipo_camada', 'dimensao_analitica', 'tipo_relacao_variavel', 'codigo_variavel', 'nivel_criticidade', 'observacao', 'aproximacao_variavel', 'nota_analista', 'cod_loteamento', 'nivel_confianca_loteamento', 'papel_no_ciclo', 'id_caso_pressao', 'entra_ipst']

Periodo coberto:
  Início : 07/04/2026
  Fim    : 2026-05-10
  Edições: 96

Amostra (5 registros):


,id_evento,data_publicacao,titulo,tipo_camada,nivel_criticidade
0,TL_20260103_001,2026-01-03,Moradores de Hortolandia voltam a protestar co...,PRESSAO_SOCIAL,alta
1,TL_20260103_002,2026-01-03,Sabesp aplica nova tarifa de agua e esgoto em ...,PRESSAO_SOCIAL,media
2,TL_20260103_003,2026-01-03,Policia Civil apura morte de bebe de um mes no...,PRESSAO_SOCIAL,alerta
3,TL_20260103_004,2026-01-03,Tarifas do transporte metropolitano sofrem rea...,IVS,media
4,TL_20260103_005,2026-01-03,Charge - Salario minimo passou a ser R$ 1.621,IVS,baixa


In [4]:
# ── Eventos por data de publicação ───────────────────────────────────────────
print("Eventos por data (top 20):")
df_total['data_publicacao'].value_counts().sort_index().tail(20)

Eventos por data (top 20):


data_publicacao
2026-04-16    3
2026-04-17    4
2026-04-18    4
2026-04-19    3
2026-04-21    3
2026-04-23    5
2026-04-24    4
2026-04-25    3
2026-04-26    5
2026-04-28    9
2026-04-29    8
2026-04-30    5
2026-05-01    8
2026-05-03    4
2026-05-05    7
2026-05-06    7
2026-05-07    4
2026-05-08    5
2026-05-09    5
2026-05-10    5
Name: count, dtype: int64

In [5]:
# ── Distribuição por tipo_camada ──────────────────────────────────────────────
print("Distribuição por tipo_camada:")
df_total['tipo_camada'].value_counts()

Distribuição por tipo_camada:


tipo_camada
GOVERNANCA          239
PRESSAO_SOCIAL      123
CONTEXTO             77
IVS                  15
nao_identificado      1
REVISAR(local)        1
Name: count, dtype: int64

In [6]:
# ── Distribuição por dimensao_analitica ───────────────────────────────────────
print("Dimensões analíticas:")
df_total['dimensao_analitica'].value_counts()

Dimensões analíticas:


dimensao_analitica
capital_humano           221
infraestrutura_urbana    139
renda_trabalho            56
multidimensional          10
CH                         9
governanca                 7
SMIDS_EXT                  4
IU                         4
RT                         3
PRESSAO_SOCIAL             1
Name: count, dtype: int64

In [7]:
# ── Distribuição por nivel_criticidade ────────────────────────────────────────
print("Nível de criticidade:")
df_total['nivel_criticidade'].value_counts()

Nível de criticidade:


nivel_criticidade
media        151
baixa        128
alerta        94
alta          79
critico        3
SMIDS_EXT      1
Name: count, dtype: int64

In [8]:
# ── Distribuição por tipo_relacao_variavel ────────────────────────────────────
print("Tipo de relação com variável IVS:")
df_total['tipo_relacao_variavel'].value_counts()

Tipo de relação com variável IVS:


tipo_relacao_variavel
indireta                 208
direta                   166
contextual                63
latente                   18
infraestrutura_urbana      1
Name: count, dtype: int64

In [9]:
# ── Distribuição por papel_no_ciclo ───────────────────────────────────────────
print("Papel no ciclo de pressão:")
df_total['papel_no_ciclo'].value_counts()

Papel no ciclo de pressão:


papel_no_ciclo
nao_aplicavel                                                  250
agravamento                                                     60
resposta                                                        50
emergencia                                                      49
REVISAR                                                         27
sinal_desfecho                                                  11
dado_auditavel                                                   1
nova ETE selada prometida 2027                                   1
IML Americana                                                    1
impacto coletivo em trabalhadores pendulares de renda baixa      1
Name: count, dtype: int64

In [10]:
# ── Ciclos de pressão ativos ──────────────────────────────────────────────────
ciclos = (
    df_total[df_total['id_caso_pressao'].notna() & (df_total['id_caso_pressao'] != '')]
    .groupby('id_caso_pressao')
    .agg(
        n_registros=('id_evento', 'count'),
        ultimo_evento=('data_publicacao', 'max'),
        ultimo_papel=('papel_no_ciclo', 'last')
    )
    .sort_values('ultimo_evento', ascending=False)
)
print("Ciclos identificados no corpus:")
ciclos

Ciclos identificados no corpus:


,n_registros,ultimo_evento,ultimo_papel
id_caso_pressao,,,
IU_AGUA_SABESP_2026,13,2026-05-10,resposta
CH_SUPERLOTACAO_CARCERARIA_2026,2,2026-05-10,resposta
CH_SAUDE_MENTAL_SITUACAO_RUA_2026,2,2026-05-10,resposta
CH_VIOLENCIA_CRIANCA_2026,5,2026-05-07,agravamento
CH_VIOLENCIA_GENERO_2025,17,2026-05-06,resposta
...,...,...,...
IU_ESCASSEZ_HIDRICA_2025,1,2026-01-04,agravamento
CH_INFLUENZA_REGIONAL_2025,1,2026-01-04,agravamento
CH_FILA_SUS_2025,1,2026-01-04,resposta


In [11]:
# ── Distribuição por município ────────────────────────────────────────────────
print("Distribuição por município:")
df_total['municipio'].value_counts()

Distribuição por município:


municipio
Hortolândia                                                                                                                                                                                                                                                                                                                                                                        271
Hortolandia                                                                                                                                                                                                                                                                                                                                                                        126
Regional                                                                                                                                                                                                                                        

In [12]:
# ── Eventos que entram no IPST-H ─────────────────────────────────────────────
print("Eventos que alimentam o IPST-H (entra_ipst = sim):")
ipst = df_total[df_total['entra_ipst'].str.lower() == 'sim']
print(f"Total: {len(ipst)}")
ipst[['data_publicacao','titulo','tipo_camada','dimensao_analitica','papel_no_ciclo','id_caso_pressao']]

Eventos que alimentam o IPST-H (entra_ipst = sim):
Total: 60


,data_publicacao,titulo,tipo_camada,dimensao_analitica,papel_no_ciclo,id_caso_pressao
0,2026-01-03,Moradores de Hortolandia voltam a protestar co...,PRESSAO_SOCIAL,infraestrutura_urbana,agravamento,IU_ETE_JATOBA_2025
1,2026-01-03,Sabesp aplica nova tarifa de agua e esgoto em ...,PRESSAO_SOCIAL,renda_trabalho,agravamento,RT_TARIFAS_SABESP_2026
3,2026-01-03,Tarifas do transporte metropolitano sofrem rea...,IVS,infraestrutura_urbana,agravamento,IU_TRANSPORTE_METROPOLITAN_2025
6,2026-01-03,Homem morre depois de atropelamento na Rodovia...,PRESSAO_SOCIAL,capital_humano,emergencia,CH_MORTES_TRANSITO_REGIONAL_2025
8,2026-01-04,Justica manda reu para Tribunal do Juri por ma...,PRESSAO_SOCIAL,capital_humano,agravamento,CH_VIOLENCIA_GENERO_2025
11,2026-01-04,Regiao encerra 2025 com aumento de mortes e ca...,PRESSAO_SOCIAL,capital_humano,agravamento,CH_INFLUENZA_REGIONAL_2025
12,2026-01-04,Populacao deve redobrar cuidados contra Aedes ...,PRESSAO_SOCIAL,capital_humano,agravamento,CH_DENGUE_2026
13,2026-01-04,Apos acoes de preservacao; Sistema Cantareira ...,PRESSAO_SOCIAL,infraestrutura_urbana,agravamento,IU_ESCASSEZ_HIDRICA_2025
19,2026-01-07,Feminicidio de Rayana Raissa Albuquerque de Ma...,PRESSAO_SOCIAL,capital_humano,agravamento,CH_VIOLENCIA_GENERO_2025
20,2026-01-07,Omissao GOV_ESTADUAL: 70% do orcamento para po...,PRESSAO_SOCIAL,capital_humano,agravamento,CH_VIOLENCIA_GENERO_2025


In [13]:
# ── Eventos IVS com relação direta (dado auditável) ──────────────────────────
ivs_direto = df_total[
    (df_total['tipo_camada'] == 'IVS') &
    (df_total['tipo_relacao_variavel'] == 'direta')
][['data_publicacao','titulo','codigo_variavel','nivel_criticidade','nota_analista']]

print(f"Eventos IVS / relação direta: {len(ivs_direto)}")
ivs_direto

Eventos IVS / relação direta: 9


,data_publicacao,titulo,codigo_variavel,nivel_criticidade,nota_analista
3,2026-01-03,Tarifas do transporte metropolitano sofrem rea...,SMIDS_EXT,media,"Vigencia 6jan2026. Hortolandia-Campinas: R$6,5..."
4,2026-01-03,Charge - Salario minimo passou a ser R$ 1.621,RT_01,baixa,Salario minimo nacional jan/2026: R$1.621. Inp...
141,2026-02-05,Monte Sinai — 2.300 moradores em ocupação info...,IU_01,alta,Monte Sinai: ocupação irregular de 20+ anos. A...
214,2026-02-24,Cetesb aplica nova multa à Sabesp por irregula...,IU_01,alta,"Cetesb multou Sabesp em R$29.967,60 (780 UFESP..."
236,2026-03-01,Tarifas do transporte metropolitano sofrem rea...,IU_03,media,"Hortolândia-Campinas passa a R$6,50 a partir 0..."
242,2026-03-04,ETE de Hortolandia acumulou 18 multas da Cetes...,IU_01,alerta,ETE de Hortolandia acumulou 18 multas da Cetes...
250,2026-03-05,Emprego formal fica positivo mas cai 43% nas c...,RT_02,media,Caged jan/2026. Hortolândia: +140 vagas formai...
366,2026-04-18,Gasolina sobe mais de 5% em Sumare e motorista...,RT_01,media,Dados oficiais ANP. Jan R$5.63 - Fev R$5.70 - ...
450,2026-05-09,Arsesp confirma má qualidade da água distribuí...,IU_01,alerta,Arsesp detectou odor e gosto em fiscalização d...


In [14]:
# ── PRESSAO_SOCIAL com nivel_criticidade = critico ou alerta ─────────────────
pressao_alta = df_total[
    (df_total['tipo_camada'] == 'PRESSAO_SOCIAL') &
    (df_total['nivel_criticidade'].isin(['critico', 'alerta']))
][['data_publicacao','municipio','localidade','titulo','papel_no_ciclo','id_caso_pressao']]

print(f"Eventos de alta pressão: {len(pressao_alta)}")
pressao_alta.sort_values('data_publicacao', ascending=False).head(20)

Eventos de alta pressão: 63


,data_publicacao,municipio,localidade,titulo,papel_no_ciclo,id_caso_pressao
453,2026-05-09,Hortolândia,Jardim Santa Clara do Lago I,"PM apreende arma calibre 9mm, réplica de fuzil...",emergencia,NaN
449,2026-05-08,Hortolandia,nao_identificado,Motociclista fica gravemente ferida apos colis...,emergencia,NaN
442,2026-05-07,Hortolandia,nao_identificado,Prefeitura de Hortolandia escala pressao sobre...,agravamento,IU_AGUA_SABESP_2026
441,2026-05-07,Hortolandia,nao_identificado,Kemilly Beatriz de 11 anos: Policia Civil inst...,agravamento,CH_VIOLENCIA_CRIANCA_2026
440,2026-05-06,Hortolandia,nao_identificado,Idosa de 78 anos morre apos atropelamento e pr...,emergencia,RT_VULNERABILIDADE_IDOSO_2026
434,2026-05-06,Hortolandia,nao_identificado,Menina de 11 anos morre apos colisao entre mot...,agravamento,CH_VIOLENCIA_CRIANCA_2026
430,2026-05-05,Hortolândia,Regional,Número de moradores em situação de rua tem que...,emergencia,CH_SAUDE_MENTAL_SITUACAO_RUA_2026
427,2026-05-05,Hortolândia,Jardim_Conceição,Jovem é achada morta em imóvel após família vo...,agravamento,CH_VIOLENCIA_GENERO_2025
423,2026-05-03,Hortolândia,Jardim_Amanda_I,Delegada destaca evidências digitais no centro...,agravamento,CH_VIOLENCIA_CRIANCA_2026
419,2026-05-01,Hortolândia,Jardim_Amanda_I,Polícia Civil abre novo inquérito sobre assass...,agravamento,CH_VIOLENCIA_CRIANCA_2026


In [15]:
# ── Exportar base consolidada ─────────────────────────────────────────────────
output_path = r'C:\Users\ailto\Atlas-Social-de-Hortolandia\outputs\tabelas\corpus_consolidado.csv'
os.makedirs(os.path.dirname(output_path), exist_ok=True)
df_total.to_csv(output_path, index=False, encoding='utf-8')
print(f"✅ Exportado: {output_path}")
print(f"   Eventos: {len(df_total)} | Colunas: {len(df_total.columns)}")

✅ Exportado: C:\Users\ailto\Atlas-Social-de-Hortolandia\outputs\tabelas\corpus_consolidado.csv
   Eventos: 456 | Colunas: 20
